In [0]:
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("domain_name", "")
dbutils.widgets.text("sub_domain_name", "")

table_name = dbutils.widgets.get("table_name").strip()
domain_name = dbutils.widgets.get("domain_name").strip()
sub_domain_name = dbutils.widgets.get("sub_domain_name").strip()

# Used only for case-insensitive metadata lookup
table_name_lower = table_name.lower()
domain_name_lower = domain_name.lower()
sub_domain_name_lower = sub_domain_name.lower()

print("INPUT")
print(f"table_name     : {table_name}")
print(f"domain_name    : {domain_name}")
print(f"sub_domain    : {sub_domain_name}")

INPUT
table_name     : 01_TXN_transaction_lifecycle
domain_name    : TXN_PROCESSING
sub_domain    : TRANSACTION_LIFECYCLE_STATUS


In [0]:
from pyspark.sql import functions as F

CATALOG = "mastercard_card_241614"
SCHEMA = "default"

SCHEMA_VALIDATION_NOTEBOOK = "schema_validation"
RULE_EXECUTION_NOTEBOOK = "rule_execution"

print("Configuration loaded")

Configuration loaded


In [0]:
if not table_name:
    raise Exception("table_name is required.")

if not domain_name:
    raise Exception("domain_name is required.")

if not sub_domain_name:
    raise Exception("sub_domain_name is required.")

print("Input validation PASSED")

Input validation PASSED


In [0]:
assets_df = spark.table(
    f"{CATALOG}.{SCHEMA}.assets_m"
)

domains_df = spark.table(
    f"{CATALOG}.{SCHEMA}.domains_m"
)

sub_domains_df = spark.table(
    f"{CATALOG}.{SCHEMA}.sub_domains_m"
)

In [0]:
asset_df = (
    assets_df
    .filter(
        F.col("is_active") == True
    )
    .filter(
        F.lower(
            F.regexp_extract(
                F.col("s3_path"),
                r"([^/]+)$",
                1
            )
        ) == table_name_lower
    )
)

In [0]:
asset_count = asset_df.count()

if asset_count == 0:
    raise Exception(
        f"No active asset found for table_name='{table_name}'."
    )

if asset_count > 1:
    raise Exception(
        f"Multiple active assets found for table_name='{table_name}'."
    )

asset = asset_df.first()

asset_id = asset["asset_id"]
asset_domain_id = asset["domain_id"]
asset_sub_domain_id = asset["sub_domain_id"]

print("ASSET RESOLVED")
print(f"asset_id         : {asset_id}")
print(f"asset_name       : {asset['asset_name']}")
print(f"domain_id        : {asset_domain_id}")
print(f"sub_domain_id    : {asset_sub_domain_id}")
print(f"s3_path          : {asset['s3_path']}")

ASSET RESOLVED
asset_id         : 28
asset_name       : 01_TXN_transaction_lifecycle
domain_id        : 1
sub_domain_id    : 1
s3_path          : source/TXN_PROCESSING/TRANSACTION_LIFECYCLE_STATUS/01_txn_transaction_lifecycle


In [0]:
domain_df = (
    domains_df
    .filter(
        F.lower(F.col("domain_name")) == domain_name_lower
    )
    .filter(
        F.col("is_active") == True
    )
)

if domain_df.count() != 1:
    raise Exception(
        f"Domain '{domain_name}' could not be uniquely resolved."
    )

domain = domain_df.first()

domain_id = domain["domain_id"]

sub_domain_df = (
    sub_domains_df
    .filter(
        F.col("domain_id") == domain_id
    )
    .filter(
        F.lower(F.col("sub_domain_name")) == sub_domain_name_lower
    )
    .filter(
        F.col("is_active") == True
    )
)

if sub_domain_df.count() != 1:
    raise Exception(
        f"Sub-domain '{sub_domain_name}' could not be uniquely resolved "
        f"under domain '{domain_name}'."
    )

sub_domain = sub_domain_df.first()

sub_domain_id = sub_domain["sub_domain_id"]

In [0]:
if asset_domain_id != domain_id:
    raise Exception(
        f"Domain mismatch. "
        f"Asset domain_id={asset_domain_id}, "
        f"requested domain_id={domain_id}."
    )

if asset_sub_domain_id != sub_domain_id:
    raise Exception(
        f"Sub-domain mismatch. "
        f"Asset sub_domain_id={asset_sub_domain_id}, "
        f"requested sub_domain_id={sub_domain_id}."
    )

print("ASSET / DOMAIN VALIDATION PASSED")
print(f"domain_id       : {domain_id}")
print(f"sub_domain_id   : {sub_domain_id}")

ASSET / DOMAIN VALIDATION PASSED
domain_id       : 1
sub_domain_id   : 1


In [0]:
schema_validation_result = dbutils.notebook.run(
    SCHEMA_VALIDATION_NOTEBOOK,
    0,
    {
        "table_name": table_name,
        "domain_name": domain_name,
        "sub_domain_name": sub_domain_name
    }
)

print(
    f"Schema validation result: {schema_validation_result}"
)

Schema validation result: PASS


In [0]:
if schema_validation_result != "PASS":
    raise Exception(
        f"Schema validation did not return PASS. "
        f"Result='{schema_validation_result}'."
    )

print("SCHEMA VALIDATION PASSED")
print("Proceeding to DQ rule execution.")

SCHEMA VALIDATION PASSED
Proceeding to DQ rule execution.


In [0]:
rules_df = spark.table(
    f"{CATALOG}.{SCHEMA}.rules_m"
)

rule_versions_df = spark.table(
    f"{CATALOG}.{SCHEMA}.rule_versions_m"
)

assignments_df = spark.table(
    f"{CATALOG}.{SCHEMA}.asset_rule_assignments_m"
)

asset_columns_df = spark.table(
    f"{CATALOG}.{SCHEMA}.asset_columns_m"
)

In [0]:
rules_to_execute_df = (
    assignments_df.alias("a")

    .join(
        rules_df.alias("r"),
        F.col("a.rule_id") == F.col("r.rule_code"),
        "inner"
    )

    .join(
        rule_versions_df.alias("rv"),
        F.col("a.rule_version_id") == F.col("rv.rule_version_id"),
        "inner"
    )

    .join(
        asset_columns_df.alias("ac"),
        (
            (F.col("a.target_column_id") == F.col("ac.column_id"))
            &
            (F.col("a.asset_id") == F.col("ac.asset_id"))
        ),
        "left"
    )

    .filter(
        F.col("a.asset_id") == asset_id
    )

    .filter(
        F.col("a.is_enabled") == True
    )

    .filter(
        F.col("r.is_active") == True
    )

    .filter(
        F.col("rv.is_active") == True
    )

    .select(
        F.col("a.assignment_id"),
        F.col("a.asset_id"),

        F.col("a.rule_id").alias("rule_code"),
        F.col("a.rule_version_id"),

        F.col("a.target_column_id"),
        F.col("ac.column_name").alias("target_column_name"),

        F.col("r.dimension"),
        F.col("r.rule_name"),
        F.col("r.rule_type"),
        F.col("r.target_type"),

        F.col("rv.version_number"),
        F.col("rv.rule_definition_parameters"),
        F.col("rv.sql_expression"),
        F.col("rv.threshold_value"),
        F.col("rv.evaluation_operation")
    )
)

In [0]:
rule_count = rules_to_execute_df.count()

print(f"ACTIVE RULES FOUND: {rule_count}")

ACTIVE RULES FOUND: 23


In [0]:
display(
    rules_to_execute_df.orderBy(
        "dimension",
        "rule_code",
        "version_number"
    )
)

assignment_id,asset_id,rule_code,rule_version_id,target_column_id,target_column_name,dimension,rule_name,rule_type,target_type,version_number,rule_definition_parameters,sql_expression,threshold_value,evaluation_operation
84,28,C01,6,254,null,COMPLETENESS,Completeness %,SQL,COLUMN / TABLE,1,null,null,95.000000,>=
62,28,C01,6,255,null,COMPLETENESS,Completeness %,SQL,COLUMN / TABLE,1,null,null,95.000000,>=
10,28,C05,10,null,null,COMPLETENESS,Record completeness %,PARAMETERIZED,TABLE,1,"{ ""parameters"":[ { ""name"":""required_columns"", ""label"":""Required columns"", ""control"":""column_multi_select"", ""required"":true } ] }",null,95.000000,>=
12,28,C07,12,null,null,COMPLETENESS,Selected-column completeness %,PARAMETERIZED,TABLE,1,"{ ""parameters"":[ { ""name"":""selected_columns"", ""label"":""Selected columns"", ""control"":""column_multi_select"", ""required"":true } ] }",null,95.000000,>=
88,28,K03,15,251,null,CONSISTENCY,Standardization compliance %,PARAMETERIZED,COLUMN,1,"{ ""parameters"":[ { ""name"":""canonical_representation"", ""label"":""Canonical representation"", ""control"":""text"", ""required"":true } ] }",null,98.000000,>=
66,28,K03,15,253,null,CONSISTENCY,Standardization compliance %,PARAMETERIZED,COLUMN,1,"{ ""parameters"":[ { ""name"":""canonical_representation"", ""label"":""Canonical representation"", ""control"":""text"", ""required"":true } ] }",null,98.000000,>=
57,28,X03,57,null,null,CUSTOM,Custom Tabular SQL Rule,CUSTOM_SQL,TABLE,1,null,null,null,null
58,28,X04,58,null,null,CUSTOM,Custom SQL Single-Result Rule,CUSTOM_SQL,TABLE / COLUMN,1,null,null,null,null
13,28,I04,13,null,null,INTEGRITY,Primary-key null %,PARAMETERIZED,TABLE,1,"{ ""parameters"":[ { ""name"":""primary_key_column"", ""label"":""Primary key column"", ""control"":""column_select"", ""required"":true } ] }",null,0.000000,<=
19,28,P01,19,null,null,PROFILING,Row count,SQL,TABLE,1,null,null,null,null


In [0]:
if rule_count == 0:

    print(
        f"No active DQ rules are assigned to asset_id={asset_id}."
    )

    dbutils.notebook.exit("NO_RULES")

In [0]:
for rule in rules_to_execute_df.collect():

    result = dbutils.notebook.run(
        RULE_EXECUTION_NOTEBOOK,
        0,
        {
            "asset_id": str(rule["asset_id"]),
            "assignment_id": str(rule["assignment_id"]),
            "rule_code": rule["rule_code"],
            "rule_version_id": str(rule["rule_version_id"]),
            "target_column_id": (
                str(rule["target_column_id"])
                if rule["target_column_id"] is not None
                else ""
            ),
            "threshold": (
                str(rule["threshold_value"])
                if rule["threshold_value"] is not None
                else ""
            ),
            "parameters": (
                rule["rule_definition_parameters"]
                if rule["rule_definition_parameters"] is not None
                else ""
            )
        }
    )

---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-7304211177227253>, line 3
      1 for rule in rules_to_execute_df.collect():
----> 3     result = dbutils.notebook.run(
      4         RULE_EXECUTION_NOTEBOOK,
      5         0,
      6         {
      7             "asset_id": str(rule["asset_id"]),
      8             "assignment_id": str(rule["assignment_id"]),
      9             "rule_code": rule["rule_code"],
     10             "rule_version_id": str(rule["rule_version_id"]),
     11             "target_column_id": (
     12                 str(rule["target_column_id"])
     13                 if rule["target_column_id"] is not None
     14                 else ""
     15             ),
     16             "threshold": (
     17                 str(rule["threshold_value"])
     18                 if rule["threshold_value"] is not None
     19                 else 

In [0]:
# assets_df = assets_df.filter(
#     F.lower(F.col("asset_name")) == table_name_lower
# )

# assets_df = assets_df.filter(
#     F.col("is_active") == True
# )

# assets_df.display()

asset_id,domain_id,sub_domain_id,asset_name,asset_type,description,is_active,created_at,updated_at,onboarded_by,onboarded_at,s3_path
28,1,1,01_TXN_transaction_lifecycle,TABLE,TXN_PROCESSING.TRANSACTION_LIFECYCLE_STATUS,true,2026-09-30T11:58:17.419Z,2026-10-01T09:48:45.630Z,1,2026-10-01T09:48:45.630Z,source/TXN_PROCESSING/TRANSACTION_LIFECYCLE_STATUS/01_txn_transaction_lifecycle
